# Semana 5 · Sesión 2: Ecuaciones y ecuaciones diferenciales

**Módulo 1**

## Objetivos de la sesión

1. Resolver ecuaciones y sistemas con `solve`, `solveset`, `linsolve` y
   `nonlinsolve`, sabiendo qué devuelve cada uno y cuándo elegirlo.
2. Resolver ecuaciones diferenciales ordinarias con `dsolve` e imponer
   condiciones iniciales con `ics`.
3. Verificar toda solución sustituyéndola de vuelta en la ecuación.

## Retomamos

En la sesión 1 derivamos, integramos, tomamos límites y desarrollamos en
serie. Con eso sabemos pasar de un potencial a su fuerza.

Hoy cerramos el círculo: de la fuerza a la **trayectoria**. Para eso hacen
falta dos cosas — resolver ecuaciones (dónde se anula algo, cuánto vale un
parámetro) y resolver ecuaciones diferenciales (cómo evoluciona un
sistema).

Este notebook es autocontenido: la celda de abajo declara todo lo que
necesita.

In [ ]:
import sympy as sp

sp.init_printing()

x, y, t, theta = sp.symbols("x y t theta", real=True)
m, k, g, L, v0 = sp.symbols("m k g L v0", positive=True)
m1, m2 = sp.symbols("m1 m2", positive=True)

## `solve`: la convención del "= 0"

`sp.solve(expresion, incognita)` resuelve suponiendo que la expresión
**está igualada a cero**. Es la convención de todo el álgebra
computacional, y ahorra escribir la mitad de las ecuaciones.

Si prefieres escribir la ecuación completa, `sp.Eq(izquierda, derecha)`
—que ya vimos en la semana 4— también le sirve. Las dos formas de abajo son
la misma pregunta.

Lo que devuelve es una **lista de Python** con las soluciones.

In [ ]:
print("solve(x**2 - 4, x)        :", sp.solve(x**2 - 4, x))
print("solve(Eq(x**2, 4), x)     :", sp.solve(sp.Eq(x**2, 4), x))

# Con símbolos en lugar de números, la respuesta es una fórmula:
print("solve(k*x**2 - m*g, x)    :")
display(sp.solve(k*x**2 - m*g, x))

## Depuración en vivo: las suposiciones también mandan aquí

Antes de ejecutar la celda, predice: ¿qué devuelve resolver $x^2 + 1 = 0$?

Recuerda que nuestra `x` se declaró con `real=True`.

In [ ]:
print("con x real   :", sp.solve(x**2 + 1, x))

x_compleja = sp.Symbol("x")           # sin suposiciones: puede ser compleja
print("sin suponer  :", sp.solve(x_compleja**2 + 1, x_compleja))

Con `x` real la lista sale **vacía**, y está bien: en los reales esa
ecuación no tiene solución. Sin suposiciones aparecen $\pm i$.

Una lista vacía no significa "SymPy no pudo". Significa "no hay soluciones
**del tipo que pediste**". Es la misma lección de la semana 4 desde otro
ángulo: las suposiciones son parte del enunciado del problema, no un
adorno.

## `solveset`: cuando las soluciones son infinitas

`solve` devuelve una lista, y una lista no puede tener infinitos elementos.
Con $\sin x = 0$ eso se nota: `solve` entrega solo unas cuantas soluciones
representativas, no todas.

`solveset` devuelve un **conjunto**, y los conjuntos de SymPy sí saben
describir familias infinitas. A cambio, su respuesta es menos cómoda de
recorrer con un `for`.

Acepta además un `domain`, para acotar dónde buscar.

In [ ]:
print("solve    :", sp.solve(sp.sin(x), x))     # solo algunas
print("solveset :")
display(sp.solveset(sp.sin(x), x))              # todas, como familia

# Acotar el dominio de búsqueda:
print("en los reales:")
display(sp.solveset(x**2 + 1, x, domain=sp.S.Reals))

## Cuál usar

| Función | Para | Devuelve |
|---|---|---|
| `sp.solve` | Una ecuación o un sistema, uso general | Lista (o lista de diccionarios) |
| `sp.solveset` | Una ecuación, cuando las soluciones son infinitas o importa el dominio | Conjunto |
| `sp.linsolve` | Sistemas **lineales** | Conjunto con una tupla |
| `sp.nonlinsolve` | Sistemas **no lineales** | Conjunto de tuplas |

**Regla práctica del curso:** empieza con `solve`, que es el más cómodo y
el que devuelve algo que puedes indexar. Cambia a los otros cuando `solve`
se quede corto: `solveset` si la respuesta es una familia infinita,
`linsolve` si el sistema es lineal (es más rápido y más predecible), y
`nonlinsolve` si no lo es.

## TODO en clase 1

La altura de un proyectil lanzado desde el suelo con rapidez $v_0$ y
ángulo $\theta$ es

$$y(t) = v_0 \sin\theta\; t - \tfrac{1}{2} g t^2$$

1. Escribe `altura` (usa `sp.Rational(1, 2)`).
2. Resuelve `altura = 0` para el tiempo. Van a salir **dos** soluciones:
   explica qué instante representa cada una.
3. Toma la que no es cero como `tiempo_de_vuelo`, y calcula el **alcance**
   sustituyéndola en la posición horizontal $x(t) = v_0\cos\theta\; t$.
4. Ciérralo con `sp.trigsimp` (semana 4). Debe quedarte
   $R = v_0^2 \sin(2\theta)/g$ — la fórmula que la semana pasada
   *escribimos*, y que hoy *deducimos*.

In [ ]:
# TODO en clase: tiempo de vuelo y alcance de un proyectil
altura = ...

tiempos = ...

tiempo_de_vuelo = ...

alcance = ...

## `linsolve`: sistemas lineales

Para un sistema lineal, `linsolve` recibe la lista de ecuaciones y la lista
de incógnitas.

El ejemplo es la **máquina de Atwood**: dos masas colgando de una polea
ideal, unidas por una cuerda. Si $m_1$ baja con aceleración $a$, la segunda
sube con la misma, y la segunda ley aplicada a cada masa da

$$m_1 a = m_1 g - T, \qquad m_2 a = T - m_2 g$$

Dos ecuaciones, dos incógnitas: la aceleración $a$ y la tensión $T$.

In [ ]:
a, T = sp.symbols("a T", real=True)

ecuaciones = [
    sp.Eq(m1*a, m1*g - T),
    sp.Eq(m2*a, T - m2*g),
]

solucion_atwood = sp.linsolve(ecuaciones, [a, T])
display(solucion_atwood)

# El conjunto trae una sola tupla; para desempaquetarla hay que sacarla:
aceleracion, tension = next(iter(solucion_atwood))

print("aceleración:")
display(aceleracion)
print("tensión:")
display(tension)

Vale la pena leer el resultado como física y no solo como álgebra:
$a = \dfrac{(m_1 - m_2)g}{m_1 + m_2}$ se anula cuando las masas son
iguales, y tiende a $g$ cuando una es mucho mayor que la otra. Los dos
casos límite son los que uno esperaría, y comprobarlos —con `subs` o con
`limit`, que ya sabes usar— es la forma barata de detectar un error de
signo.

Para sistemas **no lineales** la función es `nonlinsolve`, con la misma
firma. Aquí, la intersección de una circunferencia con una recta:

In [ ]:
display(sp.nonlinsolve([x**2 + y**2 - 25, y - x - 1], [x, y]))

## `dsolve`: ecuaciones diferenciales

Para plantear una ecuación diferencial hace falta un objeto nuevo: una
**función incógnita**, que se construye con `sp.Function`.

La diferencia con un `Symbol` es que una función se *aplica*: `x(t)` es lo
que representa la posición, y `x(t).diff(t)` su derivada. A partir de ahí,
`sp.dsolve(ecuacion, funcion)` resuelve.

Empezamos por el oscilador armónico, $\ddot{x} = -\omega^2 x$:

In [ ]:
omega = sp.Symbol("omega", positive=True)
posicion = sp.Function("x")

oscilador = sp.Eq(posicion(t).diff(t, 2), -omega**2 * posicion(t))

display(oscilador)
display(sp.dsolve(oscilador, posicion(t)))

Salen las constantes `C1` y `C2`, y tenían que salir: una ecuación de
segundo orden tiene una **familia** de soluciones de dos parámetros. La
física no está completa hasta decir de dónde parte el sistema.

Fíjate también en que `dsolve` devuelve una **ecuación** (`Eq`), no una
expresión. Para quedarte con la solución hay que pedirle el lado derecho:
`.rhs`.

## Condiciones iniciales: `ics`

`ics` es un diccionario que impone los valores iniciales. La única parte
con truco es la condición sobre la derivada, que se escribe
`funcion(t).diff(t).subs(t, 0)`: primero se deriva, y solo después se
evalúa en $t=0$.

Con una masa soltada desde la amplitud $A$ y en reposo —$x(0) = A$,
$\dot{x}(0) = 0$— las constantes quedan fijadas y sobrevive un solo coseno.

In [ ]:
A = sp.Symbol("A", positive=True)

solucion = sp.dsolve(
    oscilador,
    posicion(t),
    ics={
        posicion(0): A,                          # x(0) = A
        posicion(t).diff(t).subs(t, 0): 0,       # x'(0) = 0
    },
)

display(solucion)
display(solucion.rhs)   # solo la expresión, sin el "x(t) ="

## Verificar siempre: `checkodesol`

Una solución de una ecuación diferencial es fácil de verificar y difícil de
adivinar: basta sustituirla de vuelta y comprobar que la ecuación se
cumple. `sp.checkodesol` hace exactamente eso, y devuelve una tupla
`(True, 0)` cuando el residuo es cero.

Hazlo siempre. Es la costumbre que separa usar un CAS de creerle a un CAS.

In [ ]:
print(sp.checkodesol(oscilador, solucion))

# A mano es la misma idea: sustituir y ver que la diferencia se anula.
candidata = solucion.rhs
residuo = candidata.diff(t, 2) + omega**2 * candidata
print("residuo:", sp.simplify(residuo))

## TODO en clase 2

Un cuerpo cae con **fricción lineal**: además del peso, sufre una fuerza de
arrastre proporcional a la rapidez y opuesta al movimiento. La segunda ley
queda

$$m \frac{dv}{dt} = m g - b\, v, \qquad v(0) = 0$$

1. Declara `b` como símbolo positivo y `rapidez = sp.Function("v")`.
2. Escribe `ecuacion_caida` con `sp.Eq` y resuélvela con `dsolve`,
   imponiendo `ics={rapidez(0): 0}`.
3. Verifica la solución con `sp.checkodesol`.
4. Calcula la **velocidad terminal**: el límite de la solución cuando
   $t \to \infty$ (`sp.limit`, de la sesión 1). Debe salirte $mg/b$, que es
   justo la rapidez a la que el arrastre iguala al peso.
5. Desarrolla la solución en serie alrededor de $t = 0$ hasta orden 2. El
   primer término debe ser $gt$: a tiempos cortos, la caída todavía es
   libre.

Los puntos 4 y 5 son los dos regímenes de la misma solución —el de tiempos
largos y el de tiempos cortos— y los dos se leen con herramientas de la
sesión 1.

In [ ]:
# TODO en clase: caída con fricción lineal, y sus dos regímenes
b = ...
rapidez = ...

ecuacion_caida = ...

solucion_caida = ...

velocidad_terminal = ...

## TODO en clase 3

Cerramos juntando las dos sesiones de la semana. En la sesión 1 llegamos,
desarrollando el potencial en serie, a que un péndulo de ángulos pequeños
es un oscilador armónico con $\omega = \sqrt{g/L}$. Su ecuación de
movimiento es

$$\ddot{\theta} = -\frac{g}{L}\,\theta$$

1. Declara `angulo = sp.Function("theta")` y `theta_0` como símbolo
   positivo (el ángulo inicial).
2. Resuelve con `dsolve` e `ics`, soltando el péndulo desde el reposo:
   $\theta(0) = \theta_0$ y $\dot{\theta}(0) = 0$.
3. Verifica con `checkodesol`.
4. Lee la frecuencia en el argumento del coseno y escribe el **periodo**
   $T = 2\pi/\omega$. Tiene que salirte $T = 2\pi\sqrt{L/g}$ — la fórmula
   que en la semana 4 escribimos a mano en un `srepr`, y que ahora sale de
   resolver la dinámica.

Ese es el arco de las cinco semanas: la misma fórmula, primero tecleada,
después deducida.

In [ ]:
# TODO en clase: el péndulo pequeño, resuelto
angulo = ...
theta_0 = ...

ecuacion_pendulo = ...

solucion_pendulo = ...

periodo = ...

## Resumen

Hoy resolvimos. Para ecuaciones algebraicas, `solve` es el caballo de
batalla —con la convención de que la expresión está igualada a cero, y
respetando las suposiciones de los símbolos—, `solveset` aparece cuando las
soluciones son una familia infinita, y `linsolve`/`nonlinsolve` cuando hay
un sistema.

Para ecuaciones diferenciales, `sp.Function` declara la incógnita, `dsolve`
la resuelve, `ics` fija las constantes de integración y `checkodesol`
comprueba que la respuesta es de verdad una solución. Verificar es parte
del método, no un extra.

Con esto cerramos el cálculo simbólico: derivar, integrar, aproximar y
resolver. Una fuerza entra por un lado y sale una trayectoria por el otro.

**Tarea de esta semana:** [`tarea-05.ipynb`](../tarea/tarea-05.ipynb), que
se entrega por Pull Request dentro de tu fork (ver
[`docs/git-guia.md`](../../docs/git-guia.md)).

**Próxima clase — Semana 6:** álgebra lineal simbólica. `Matrix`,
eigenvalores y eigenvectores simbólicos, graficación con
`sympy.plotting`, y `latex()` para llevar los resultados a un reporte.